# Extra biosphere and iLUC import

This notebook imports the additional biosphere flows first and then imports the iLUC database.

Project names, database names and file paths are defined in `functions/utils.py`.
The preparation and linking functions are stored in `functions/iluc_import.py`.


### 1. Import required packages


In [ ]:
import bw2data as bd
import pandas as pd

from functions.iluc_import import (
    prepare_extra_biosphere_import,
    prepare_iluc_import,
)
from functions.utils import Config

### 2. Select Brightway project


In [ ]:
bd.projects.set_current(Config.PROJECT_NAME)

print(f"Current Brightway project: {bd.projects.current}")

### 3. Check input files

Both Excel workbooks are stored in the shared OneDrive database folder.


In [ ]:
assert Config.EXTRA_BIOSPHERE_PATH.is_file(), (
    f"Extra biosphere workbook not found: {Config.EXTRA_BIOSPHERE_PATH}"
)

assert Config.ILUC_PATH.is_file(), (
    f"iLUC workbook not found: {Config.ILUC_PATH}"
)

print(f"Extra biosphere: {Config.EXTRA_BIOSPHERE_PATH}")
print(f"iLUC workbook:    {Config.ILUC_PATH}")

### 4. Prepare extra biosphere import

The extra biosphere is prepared first because the iLUC database may contain elementary flows that link to it.


In [ ]:
extra_biosphere_importer = prepare_extra_biosphere_import(
    Config.EXTRA_BIOSPHERE_PATH,
    Config.PROJECT_NAME,
)

pd.DataFrame(extra_biosphere_importer.unlinked)

### 5. Write extra biosphere database


In [ ]:
assert extra_biosphere_importer.all_linked, (
    "Not all extra biosphere exchanges could be linked."
)

extra_biosphere_importer.write_database()

### 6. Prepare iLUC import

The iLUC inventory is linked to consequential ecoinvent, the main biosphere database, and the extra biosphere database.


In [ ]:
iluc_importer = prepare_iluc_import(
    source_path=Config.ILUC_PATH,
    project_name=Config.PROJECT_NAME,
    ecoinvent_database_name=Config.ECOINVENT_CLCA_NAME,
    biosphere_database_name=Config.BIOSPHERE_NAME,
    extra_biosphere_database_name=Config.BIOSPHERE_EXTRA_NAME,
    sheet_name=Config.ILUC_SHEET_NAME,
)

### 7. Check iLUC linking


In [ ]:
pd.DataFrame(iluc_importer.unlinked)

### 8. Write iLUC database


In [ ]:
assert iluc_importer.all_linked, (
    "Not all iLUC exchanges could be linked."
)

iluc_importer.write_database()

### 9. Verify the import


In [ ]:
bd.databases